# Classifying Texts using Multinomial Naive Bayes

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer

### Classification and Analysis of the Data

In [ ]:
y_eng = pd.read_csv("../Naive-Bayes/dataset/IMDB.csv", usecols=["text"])
y_eng["language"] = 0
y_fre = pd.read_csv("../Naive-Bayes/language-dataset/french_tweets.csv", usecols=["text"])
y_fre["language"] = 1
y_spn = pd.read_csv("../Naive-Bayes/language-dataset/IMDB-SPN.CSV", usecols=["text"])
y_spn["language"] = 2
# Creating Testing dataset and merging it
y_fre = y_fre.sample(50000, random_state=42)
y_eng_train, y_eng_test = train_test_split(y_eng, test_size=0.3, random_state=42)
y_fre_train, y_fre_test = train_test_split(y_fre, test_size=0.3, random_state=42)
y_spn_train, y_spn_test= train_test_split(y_spn, test_size=0.3, random_state=42)
y_train = pd.concat([y_eng_train, y_fre_train, y_spn_train], axis=0).reset_index(drop=True)
y_test = pd.concat([y_eng_test, y_fre_test, y_spn_test], axis = 0).reset_index(drop=True)
print(f"y_train:{y_train.shape}")
print(f"y_test:{y_test.shape}")

y_train:(105000, 2)
y_test:(45000, 2)


### Vectorize the data
We now vectorize the data with CountVectorizer

In [66]:
vectorizer = CountVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(y_train['text']) 
X_test_text = vectorizer.transform(y_test['text'])   
print("Vectorized the text data")

Vectorized the text data


### Defining Parameters and estimating Max Likelihood for a certain custom message

We define parameters phi_y for each custom class that exists such as English, Spanish and French.<br>
Calculating the log Maximum Likelihood i.e. probability of a certain sentence being part of any language.

In [75]:
def param_y(y):
    y_np = y.to_numpy() 
    phi_y_list = [
        np.sum(y_np == 0) / len(y_np),
        np.sum(y_np == 1) / len(y_np),
        np.sum(y_np == 2) / len(y_np)
    ]
    return phi_y_list

def params_y(y, X):
    y_np = y.to_numpy()

    X_y0 = X[y_np == 0] 
    X_y1 = X[y_np == 1]
    X_y2 = X[y_np == 2]

    phi_k_y0 = (X_y0.sum(axis=0) + 1) / (X_y0.sum() + len(vectorizer.get_feature_names_out())) 
    phi_k_y1 = (X_y1.sum(axis=0) + 1) / (X_y1.sum() + len(vectorizer.get_feature_names_out())) 
    phi_k_y2 = (X_y2.sum(axis=0) + 1) / (X_y2.sum() + len(vectorizer.get_feature_names_out())) 

    return phi_k_y0, phi_k_y1, phi_k_y2
def predict(phi_y_list, phi_k_list, X):
    if len(X.shape) == 1:
        X = X.reshape(1, -1)

    # Compute log probabilities for each class
    predictions = []
    for i in range(len(phi_y_list)):
        log_prob = np.log(phi_y_list[i]) + X.dot(np.log(phi_k_list[i]).T) # Corrected dot product and log application
        predictions.append(log_prob)

    return np.argmax(predictions, axis=0)
def classify(predicted_value):
    if predicted_value == 0:
        return "English"
    elif predicted_value == 1:
        return "French"
    else:
        return "Spanish"

### This is where the magic takes place


In [79]:

params_list = param_y(y_train["language"]) 
parameters_y_k = params_y(y_train["language"], X_train) 
print("Parameters calculated")

y_pred = predict(params_list, parameters_y_k, X_test_text)
accuracy = accuracy_score(y_test["language"], y_pred)
print(f"Accuracy on the test set: {accuracy * 100:.2f}%")

test_sentence = input("Enter a sentence in English/French/Spanish:")
X_test_single = vectorizer.transform([test_sentence]) 
predictions_single = predict(params_list, parameters_y_k, X_test_single)
predicted_language = classify(predictions_single)
print(f"Prediction for '{test_sentence}': Language ID = {predicted_language}") 

Parameters calculated
Accuracy on the test set: 98.23%
Prediction for 'I love machine learning': Language ID = English
